# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MRazaRashid/FlyRank_Week1/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I'm choosing Random Forest for my lane (Refresh/Content Opportunity Scoring). Logistic Regression is a simple starting point,but the lane guide's own starter pipeline results show tree-based methods handling this problem much better.

Decision tree already jumps from the baseline's 0.240 precision@50 to 0.540, and random forest reaches 0.740. This makes sense for my problem: the signals I have (staleness, demand, position, CTR) likely interact in non-linear ways and random forest can capture those interactions automatically, while logistic regression assumes a simpler linear relationship. I'm avoiding gradient boosting for now since the assignment flags it as "where safe" with my modest dataset size, the added complexity risks overfitting without much upside over random forest.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I'm using a client-grouped train/test split, not a plain random split. This matters because multiple pages belong to the same client, and pages from the same client likely share underlying patterns (writing style, industry, site structure) that a model could partially memorize rather than genuinely learn from signals. If pages from the same client end up in both train and test, the model's test performance would look better than it really is. A client-grouped split keeps every page from a given client entirely in either train or test, so the model is genuinely tested on clients it has never seen.

In [1]:

import os, sys, subprocess, pandas as pd
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found.")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

Working dir: /content/flyrank-ml-internship-starter
Starter data found.


In [2]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print("Clients in train:", train_df["client_id"].nunique())
print("Clients in test:", test_df["client_id"].nunique())
print("Overlapping clients (should be 0):", len(overlap))

Clients in train: 25
Clients in test: 7
Overlapping clients (should be 0): 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

feature_cols = ["impressions_90d", "content_age_days", "days_since_last_update",
                 "avg_position", "ctr", "engagement_rate", "word_count"]

In [4]:
X_train = train_df[feature_cols].fillna(0)
y_train = (train_df["trend_direction"] == "down").astype(int)
X_test = test_df[feature_cols].fillna(0)
y_test = (test_df["trend_direction"] == "down").astype(int)

In [5]:
model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200,
                       random_state=42)

In [6]:
model_probs = model.predict_proba(X_test)[:, 1]
model_auc = roc_auc_score(y_test, model_probs)
model_ap = average_precision_score(y_test, model_probs)

In [7]:
def precision_at_k(y_true, scores, k=50):
    top_k_idx = pd.Series(scores).nlargest(k).index
    return y_true.iloc[top_k_idx].mean()

model_p50 = precision_at_k(y_test.reset_index(drop=True), model_probs, k=50)

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.